In [11]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [12]:
np.random.seed(42)

n_samples = 800

# Features
age = np.random.randint(18, 60, n_samples)
income = np.random.randint(20000, 120000, n_samples)
hours_studied = np.random.randint(0, 10, n_samples)
previous_score = np.random.randint(40, 100, n_samples)
sleep_hours = np.random.uniform(4, 9, n_samples)

# Linear combination to generate probability
z = (
    0.03 * age +
    0.00002 * income +
    0.6 * hours_studied +
    0.05 * previous_score +
    0.3 * sleep_hours
    - 10
)

# Sigmoid
prob = 1 / (1 + np.exp(-z))

# Convert to class
target = (prob > 0.5).astype(int)

df = pd.DataFrame({
    "age": age,
    "income": income,
    "hours_studied": hours_studied,
    "previous_score": previous_score,
    "sleep_hours": sleep_hours,
    "pass_exam": target
})

df.to_csv("logistic_regression_dataset.csv", index=False)

print(df.head())

   age  income  hours_studied  previous_score  sleep_hours  pass_exam
0   56   24621              6              74     5.506326          1
1   46   35034              9              63     6.991787          1
2   32   25126              9              56     5.486189          1
3   25   25122              4              44     5.499596          0
4   38   38030              6              59     7.715966          1


In [13]:
def sigmoid(x):
  s = 1 / (1 + np.exp(-x))
  return s

In [14]:
def logistic_regression(dataset, unseen_dataset):
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    split=int(0.7*len(dataset))

    train_data=dataset.iloc[:split]
    test_data=dataset.iloc[split:]

    x_train=train_data.iloc[:,:-1]
    y_train=train_data.iloc[:,-1].values

    x_test=test_data.iloc[:,:-1]
    y_test=test_data.iloc[:,-1].values

    mean=x_train.mean()
    std=x_train.std()

    x_train=(x_train-mean)/std
    x_test=(x_test-mean)/std

    x_train=x_train.values
    x_test=x_test.values

    n_weights=len(dataset.columns)-1
    weights=np.random.rand(n_weights)

    learning_rate=0.01
    epochs=20

    for i in range(1,epochs+1):
        indices = np.random.permutation(len(x_train))
        x_shuffled = x_train[indices]
        y_shuffled = y_train[indices]

        for x,y in zip(x_shuffled,y_shuffled):
            z=np.dot(weights,x)
            p=sigmoid(z)
            error=p-y
            weights=weights- learning_rate*error*x
    
    predictions=[]

    for x in x_test:
        z=np.dot(weights,x)
        p=sigmoid(z)

        if p>=0.5:
            predictions.append(1)
        else:
            predictions.append(0)

    predictions=np.array(predictions)

    accuracy=np.sum(predictions==y_test)/len(y_test)
    print('Accuracy:',accuracy)

    unseen_dataset=unseen_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    unseen_dataset = (unseen_dataset - mean) / std
    unseen_dataset = unseen_dataset.values

    unseen_predictions = []

    for x in unseen_dataset:
        z = np.dot(weights, x)
        p = sigmoid(z)

        if p >= 0.5:
            unseen_predictions.append(1)
        else:
            unseen_predictions.append(0)

    return unseen_predictions



In [15]:
unseen_dataset = pd.read_csv("logistic_regression_unseen_dataset.csv")

In [16]:
preds = logistic_regression(df, unseen_dataset)
print(preds)

Accuracy: 0.9291666666666667
[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0]
